In [1]:
import pandas as pd
import numpy as np
import os
import csv

# Load data

In [2]:
input_dir = "/mnt/c/users/helen/Desktop/test"

In [35]:
dfs = []

for root, dirs, files in os.walk(input_dir):
    for filename in files:
        if filename.lower().endswith(".csv"):

            path = os.path.join(root, filename)

            df = pd.read_csv(path)

            # Optional metadata
            df["File"] = os.path.splitext(filename)[0].replace(" ", "_")
            df["Path"] = path

            dfs.append(df) 

# Combine all tables
data = pd.concat(dfs, ignore_index=True)

# Create ROI column
data['ROI'] = data['Label'].apply(lambda x: x.split(":")[1])

# Delete first 3 columns
data.drop(data.columns[[0, 1, 2]], axis=1, inplace=True)

# Reorder columns
data = data[
    [
        "File",
        "Measurement_type",
        "Length",
        "ROI",
        "Path"
    ]
]

In [36]:
# Split data into 2 dataframes
speed = data[data['Measurement_type']=='Fiber_length']
iod = data[data['Measurement_type']=='Interorigin_distance']

In [41]:
# Checking speed file
counts = speed.groupby("File").size()
odd_files = counts[counts % 2 != 0].index.tolist()
#odd_files = counts[counts % 2 == 0].index.tolist()

if len(odd_files) == 0:
    print("All files contain an even number of fibers.")
else:
    print("The following files contain an odd number of fibers will be removed:")
    print(*odd_files, sep="\n")
    
    # Removing odd files from speed dataframe
    speed = speed[~speed["File"].isin(odd_files)].copy()

All files contain an even number of fibers.


In [54]:
conversion_factor = 2.59 # kb/µm
time = 20 # minutes

In [52]:
speed["Index"] = speed.groupby("File").cumcount() // 2

speed_processed = speed.groupby(["File", "Index"], as_index=False).agg(
        Total_Length=("Length", "sum"),
        ROI=("ROI", list),
        Path=("Path", "first"),
        )

In [56]:
speed_processed['Speed_kb_min'] = speed_processed['Total_Length'].apply(lambda x: x * conversion_factor / time)

In [59]:
speed_processed

,File,Index,Total_Length,ROI,Path,Speed_kb_min
0,siORC1_MGS1-14_Fiber_length,0,243.804,"[0549-0366, 0448-0508]",/mnt/c/users/helen/Desktop/test/siORC1_MGS1-14...,31.572618
1,siORC1_MGS1-14_Fiber_length,1,220.822,"[0318-0501, 0235-0613]",/mnt/c/users/helen/Desktop/test/siORC1_MGS1-14...,28.596449
2,siORC1_MGS1-14_Fiber_length,2,287.811,"[0209-0494, 0085-0630]",/mnt/c/users/helen/Desktop/test/siORC1_MGS1-14...,37.271525
3,siORC1_MGS1-15_Fiber_length,0,321.122,"[0659-0571, 0574-0747]",/mnt/c/users/helen/Desktop/test/siORC1_MGS1-15...,41.585299


In [63]:
speed_processed = speed_processed[['File', 'Speed_kb_min', 'ROI', 'Path']]

In [64]:
speed_processed

,File,Speed_kb_min,ROI,Path
0,siORC1_MGS1-14_Fiber_length,31.572618,"[0549-0366, 0448-0508]",/mnt/c/users/helen/Desktop/test/siORC1_MGS1-14...
1,siORC1_MGS1-14_Fiber_length,28.596449,"[0318-0501, 0235-0613]",/mnt/c/users/helen/Desktop/test/siORC1_MGS1-14...
2,siORC1_MGS1-14_Fiber_length,37.271525,"[0209-0494, 0085-0630]",/mnt/c/users/helen/Desktop/test/siORC1_MGS1-14...
3,siORC1_MGS1-15_Fiber_length,41.585299,"[0659-0571, 0574-0747]",/mnt/c/users/helen/Desktop/test/siORC1_MGS1-15...


In [5]:
# Info
n_files_speed = len(set(speed['File']))
n_files_iod = len(set(iod['File']))

# Print information
print(f"The amount of files with replication speed measurements is: {n_files_speed}")
print(f"The amount of files with IOD measurements is: {n_files_iod}")

The amount of files with replication speed measurements is: 2
The amount of files with IOD measurements is: 2


In [ ]:
data

In [ ]:
iod 